# 情報基礎 対面4 ── 文の生成

ウィキニュース日本語版の記事から，直前の文字ごとに次の文字の回数を数えた表を作る．表から次の文字の確率を計算し，確率に従って次の文字を選んで末尾に足すことを繰り返して，文を生成する．

**進め方**

- 上のセルから順に，セルの左の ▶（実行ボタン）を押す．コードは書き換えない
- 2，3，4 では，セルの右の欄に文字や数を入れてから ▶ を押す
- 生成される文は，事実とは限らない
- 生成した文は，表示の前に，人名を「○○」に加工している
- `NameError` が出たら，1 のセルから順に実行し直す
- ほかのエラーが出たら，まず最後の1行を読む．エラーの種類と内容が表示される

学習データの出典：ウィキニュース日本語版（https://ja.wikinews.org/ ）の記事．CC BY 4.0．記事の本文から文を取り出し，一部の記事を除いた．

## 1 データの読み込み

次のセルを実行する．学習データ（ウィキニュース日本語版の記事の文）を読み込み，直前 1〜4文字ごとに，次の文字の回数を数えた表を作る．30〜40秒ほどかかる．

「文 19,785，文字 1,181,539，文字の種類 2,943」と表示されれば，準備は完了．

In [ ]:
!pip -q install janome
import collections
import random
import re
import urllib.request
from janome.tokenizer import Tokenizer

URL = "https://raw.githubusercontent.com/kazumichiINOUE/info-basics-notebooks/25f422f57b33f844016e28e2585529f6e6e7ddfe/data/wikinews_ja.txt"

# 学習データ：1行に1文。文と文の間の改行も，1つの文字として数える
sentences = urllib.request.urlopen(URL).read().decode("utf-8").splitlines()
text = "\n".join(sentences)


def make_table(k):
    """直前 k 文字ごとに，次の文字の回数を数えた表"""
    table = collections.defaultdict(collections.Counter)
    for i in range(len(text) - k):
        table[text[i:i + k]][text[i + k]] += 1
    return table


tables = {k: make_table(k) for k in (1, 2, 3, 4)}


def next_char(s, k):
    """文字列 s の末尾の k 文字の行から，確率に従って次の文字を選ぶ。
    その行が表に無ければ，直前の文字数を1つずつ減らして探す。選んだ文字と，使った文字数を返す"""
    for n in range(min(k, len(s)), 0, -1):
        row = tables[n].get(s[-n:])
        if row:
            return random.choices(list(row.keys()), weights=list(row.values()))[0], n
    return None, 0


def generate(k, start, limit=70):
    """確率に従って次の文字を選び，末尾に足すことを繰り返す。文の区切り（改行）が選ばれたら終える"""
    s = start
    while len(s) < limit:
        c, _ = next_char(s, k)
        if c is None or c == "\n":
            break
        s += c
    return s


# 出力のフィルター：生成した文を表示する前に，人名を「○○」にする（確率を使わない別のプログラム）
TITLE = ("(?:氏|さん|選手|監督|投手|捕手|騎手|力士|議員|首相|総理|大統領|社長|会長|教授|名人|九段|八段|七段|アナウンサー"
         "|容疑者|被告|知事|市長|大臣|長官|コーチ|オーナー|博士|君|ちゃん|様)")
RULE = re.compile("(?<![一-龥ァ-ヶー])(?!同|両|各|当|本|全|新|前|元|現|副)[一-龥ァ-ヶー・]{2,8}(?=" + TITLE + ")")
tokenizer = Tokenizer()


def mask(s):
    out = []
    for t in tokenizer.tokenize(s):
        pos = t.part_of_speech.split(",")
        out.append("○○" if (pos[0] == "名詞" and pos[1] == "固有名詞" and pos[2] == "人名") else t.surface)
    m = RULE.sub("○○", "".join(out))
    return re.sub("(○○[・＝=]?)+", "○○", m)


print(f"文 {len(sentences):,}，文字 {len(text):,}，文字の種類 {len(set(text)):,}")

## 2 次の文字の確率（課題1）

セルの右の欄に文字を1つ入れて，セルを実行する．その文字の次に来た文字の回数と確率が，確率の大きい順に10個表示される．

確率 ＝ 回数 ÷ 行の回数の和．

In [ ]:
#@markdown 文字を1つ入れる
moji = "東"  #@param {type:"string"}

ch = moji.strip()[:1]
row = tables[1].get(ch)
if not row:
    print(f"「{ch}」は，学習データに無い．ほかの文字を入れる")
else:
    total = sum(row.values())
    print(f"直前の文字「{ch}」　行の回数の和 {total:,}回　次の文字 {len(row):,}種類")
    print()
    for c, n in row.most_common(10):
        name = "（文の区切り）" if c == "\n" else f"「{c}」"
        print(f"  {name}　確率 {n / total:.3f}　{n:>6,}回　" + "■" * round(n / total * 40))
    rest = total - sum(n for _, n in row.most_common(10))
    if rest:
        print(f"  そのほか {len(row) - 10:,}種類　確率 {rest / total:.3f}　{rest:>6,}回")

## 3 生成（課題2）

セルの右の欄で直前の文字数を選び，セルを実行する．確率に従って次の文字を選び，末尾に足すことを繰り返して，5つの文字列を生成する．書き出しの4文字は，学習データの文から選ぶ．

直前の文字数を 1，2，3，4 に変えて実行し，出力を比べる．実行のたびに，出力は変わる．

In [ ]:
#@markdown 直前の文字数を選ぶ
k = 1  #@param [1, 2, 3, 4] {type:"raw"}

print(f"直前の文字数 {k}（表の行の数 {len(tables[k]):,}）")
print()
for _ in range(5):
    start = random.choice(sentences)[:4]
    print("・" + mask(generate(k, start)))
print()
print("生成した文は，事実とは限らない．表示の前に，人名を「○○」に加工している．")

## 4 自分の文からの生成（課題3）

セルの右の欄に，自分の文の書き出し（10文字くらいまで）を入れて，セルを実行する．その書き出しに続けて，直前4文字の表で5つの文を生成する．

- **実在の人物の名前は書かない**
- 書き出しの末尾の4文字が学習データに無いときは，直前の文字数を減らして次の文字を選ぶ（そのことが表示される）

In [ ]:
#@markdown 自分の文の書き出しを入れる（実在の人物の名前は書かない）
bun = "埼玉大学は"  #@param {type:"string"}

start = bun.strip()
c, n = next_char(start, 4)
if c is None:
    print("この書き出しの末尾の文字は，学習データに無い．ほかの書き出しを入れる")
else:
    if n < min(4, len(start)):
        print(f"末尾の4文字「{start[-4:]}」は学習データに無いので，はじめは直前{n}文字「{start[-n:]}」の行から次の文字を選ぶ")
        print()
    for _ in range(5):
        print("・" + mask(generate(4, start)))
    print()
    print("生成した文は，事実とは限らない．表示の前に，人名を「○○」に加工している．")